## BiLSTM-CRF
Word-level BiLSTM + CRF. No character CNN — uses FastText word embeddings only.

In [15]:
import numpy as np
import pickle
import json
import random
import itertools
from pathlib import Path
from collections import defaultdict
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, f1_score
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device  : {DEVICE}')
print(f'PyTorch : {torch.__version__}')

Device  : cpu
PyTorch : 2.12.0+cpu


### Load data

In [16]:
DATA_DIR  = Path('../../datasets/processed_data')
SPLIT_DIR = Path('../../datasets/split_data')

# Per-split token arrays (no char arrays needed)
X_tokens_train = np.load(DATA_DIR / 'X_tokens_train.npy')
Y_labels_train = np.load(DATA_DIR / 'Y_labels_train.npy')
masks_train    = np.load(DATA_DIR / 'masks_train.npy')

X_tokens_val   = np.load(DATA_DIR / 'X_tokens_val.npy')
Y_labels_val   = np.load(DATA_DIR / 'Y_labels_val.npy')
masks_val      = np.load(DATA_DIR / 'masks_val.npy')

X_tokens_test  = np.load(DATA_DIR / 'X_tokens_test.npy')
Y_labels_test  = np.load(DATA_DIR / 'Y_labels_test.npy')
masks_test     = np.load(DATA_DIR / 'masks_test.npy')

emb_mat = np.load(DATA_DIR / 'embedding_matrix.npy')

with open(DATA_DIR / 'vocabs.pkl', 'rb') as f:
    vocabs = pickle.load(f)

token2id      = vocabs['token2id']
id2token      = vocabs['id2token']
label2id      = vocabs['label2id']
id2label      = vocabs['id2label']
PAD_ID        = vocabs['PAD_ID']
UNK_ID        = vocabs['UNK_ID']
PAD_LABEL_ID  = vocabs['PAD_LABEL_ID']
Entity_labels = vocabs['Entity_labels']
MAX_LEN       = vocabs['MAX_LEN_LSTM']
EMBED_DIM     = vocabs['EMBED_DIM']

NUM_LABELS = len(label2id)

with open(SPLIT_DIR / 'split_indices.json') as f:
    split_idx = json.load(f)
idx_train          = split_idx['idx_train']
idx_val            = split_idx['idx_val']
idx_test           = split_idx['idx_test']
idx_train_balanced = split_idx['idx_train_balanced']

print(f'Train / Val / Test : {len(idx_train)} / {len(idx_val)} / {len(idx_test)}')
print(f'Train balanced     : {len(idx_train_balanced)}')

Train / Val / Test : 154 / 33 / 33
Train balanced     : 300


### Dataset & DataLoader

In [17]:
class ResumeNERDataset(Dataset):
    """Word-only dataset — no char IDs."""
    def __init__(self, X_tokens, Y_labels, masks):
        self.X_tokens = torch.tensor(X_tokens, dtype=torch.long)
        self.Y_labels = torch.tensor(Y_labels, dtype=torch.long)
        self.masks    = torch.tensor(masks,    dtype=torch.bool)

    def __len__(self):
        return len(self.X_tokens)

    def __getitem__(self, idx):
        return self.X_tokens[idx], self.Y_labels[idx], self.masks[idx]


BATCH_SIZE = 32

train_ds = ResumeNERDataset(X_tokens_train, Y_labels_train, masks_train)
val_ds   = ResumeNERDataset(X_tokens_val,   Y_labels_val,   masks_val)
test_ds  = ResumeNERDataset(X_tokens_test,  Y_labels_test,  masks_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

Train: 300 | Val: 33 | Test: 33


### Model

In [18]:
from torchcrf import CRF


class BiLSTM_CRF(nn.Module):
    """
    BiLSTM-CRF for NER — word embeddings only (no CharCNN).

    Pipeline
    --------
    Token IDs → FastText word embeddings → BiLSTM → Linear → CRF
    """

    def __init__(
        self,
        embedding_matrix : np.ndarray,
        num_tags         : int,
        pad_token_id     : int,
        pad_tag_id       : int,
        hidden_dim       : int   = 256,
        num_layers       : int   = 2,
        dropout          : float = 0.5,
        freeze_embeddings: bool  = False,
    ):
        super().__init__()
        vocab_size, embed_dim = embedding_matrix.shape

        # Word embedding
        self.word_emb = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_token_id)
        self.word_emb.weight = nn.Parameter(
            torch.tensor(embedding_matrix, dtype=torch.float32),
            requires_grad=not freeze_embeddings
        )

        self.dropout = nn.Dropout(dropout)

        # BiLSTM
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim // 2,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        # Projection → tag space
        self.fc  = nn.Linear(hidden_dim, num_tags)
        self.crf = CRF(num_tags, batch_first=True)

        self.pad_tag_id = pad_tag_id

    def _get_emissions(self, token_ids, mask):
        x = self.dropout(self.word_emb(token_ids))       # (B, L, E)
        x, _ = self.lstm(x)                              # (B, L, H)
        x = self.dropout(x)
        return self.fc(x)                                # (B, L, T)

    def forward(self, token_ids, labels, mask):
        emissions = self._get_emissions(token_ids, mask)
        loss = -self.crf(emissions, labels, mask=mask, reduction='mean')
        return loss

    @torch.no_grad()
    def decode(self, token_ids, mask):
        emissions = self._get_emissions(token_ids, mask)
        return self.crf.decode(emissions, mask=mask)

### Instantiate model

In [19]:
model = BiLSTM_CRF(
    embedding_matrix  = emb_mat,
    num_tags          = NUM_LABELS,
    pad_token_id      = PAD_ID,
    pad_tag_id        = PAD_LABEL_ID,
    hidden_dim        = 256,
    num_layers        = 2,
    dropout           = 0.5,
    freeze_embeddings = False,
).to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(model)
print(f'\nTrainable params : {trainable:,}')
print(f'Total params     : {total:,}')

BiLSTM_CRF(
  (word_emb): Embedding(3378, 300, padding_idx=0)
  (dropout): Dropout(p=0.5, inplace=False)
  (lstm): LSTM(300, 128, num_layers=2, batch_first=True, dropout=0.5, bidirectional=True)
  (fc): Linear(in_features=256, out_features=22, bias=True)
  (crf): CRF(num_tags=22)
)

Trainable params : 1,855,166
Total params     : 1,855,166


### Training helpers

In [20]:
def train_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    for token_ids, labels, mask in loader:
        token_ids = token_ids.to(device)
        labels    = labels.to(device)
        mask      = mask.to(device)
        optimizer.zero_grad()
        loss = model(token_ids, labels, mask)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def eval_epoch(model, loader, device):
    model.eval()
    total_loss = 0.0
    for token_ids, labels, mask in loader:
        token_ids = token_ids.to(device)
        labels    = labels.to(device)
        mask      = mask.to(device)
        loss = model(token_ids, labels, mask)
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def collect_predictions(model, loader, device, id2label, pad_label_id):
    model.eval()
    y_true, y_pred = [], []
    for token_ids, labels, mask in loader:
        token_ids = token_ids.to(device)
        mask_bool = mask.bool().to(device)
        preds = model.decode(token_ids, mask_bool)
        for i, length in enumerate(mask.sum(dim=1).tolist()):
            length = int(length)
            y_true.extend(labels[i, :length].tolist())
            y_pred.extend(preds[i][:length])
    y_true = [id2label.get(t, 'O') for t in y_true]
    y_pred = [id2label.get(t, 'O') for t in y_pred]
    return y_true, y_pred

### Random search

In [21]:
PARAM_DISTRIBUTIONS = {
    'hidden_dim'       : [128, 256, 512],
    'num_layers'       : [1, 2],
    'dropout'          : [0.3, 0.5],
    'freeze_embeddings': [False],
    'lr'               : [1e-3, 3e-3, 5e-4],
    'weight_decay'     : [0.0, 1e-4, 5e-4],
    'batch_size'       : [16, 32],
}

N_TRIALS    = 10
SEARCH_EPOCHS = 20
PATIENCE      = 3
Path('model_result').mkdir(exist_ok=True)
SEARCH_CKPT   = 'model_result/bilstm_crf_search.pt'


def run_trial(params, trial_id):
    t_loader = DataLoader(train_ds, batch_size=params['batch_size'], shuffle=True)
    v_loader = DataLoader(val_ds,   batch_size=params['batch_size'], shuffle=False)

    m = BiLSTM_CRF(
        embedding_matrix  = emb_mat,
        num_tags          = NUM_LABELS,
        pad_token_id      = PAD_ID,
        pad_tag_id        = PAD_LABEL_ID,
        hidden_dim        = params['hidden_dim'],
        num_layers        = params['num_layers'],
        dropout           = params['dropout'],
        freeze_embeddings = params['freeze_embeddings'],
    ).to(DEVICE)

    opt = optim.Adam(m.parameters(), lr=params['lr'], weight_decay=params['weight_decay'])
    best_val, no_imp = float('inf'), 0

    for ep in range(1, SEARCH_EPOCHS + 1):
        train_epoch(m, t_loader, opt, DEVICE)
        val_loss = eval_epoch(m, v_loader, DEVICE)
        if val_loss < best_val:
            best_val = val_loss
            torch.save(m.state_dict(), SEARCH_CKPT)
            no_imp = 0
        else:
            no_imp += 1
            if no_imp >= PATIENCE:
                break

    print(f'Trial {trial_id:2d} | val_loss={best_val:.4f} | {params}')
    return {'trial': trial_id, 'val_loss': best_val, **params}


random.seed(42)
results = []
for i in range(1, N_TRIALS + 1):
    params = {k: random.choice(v) for k, v in PARAM_DISTRIBUTIONS.items()}
    results.append(run_trial(params, i))

results_df = pd.DataFrame(results).sort_values('val_loss').reset_index(drop=True)
print('\nTop 5:')
print(results_df.head())

Trial  1 | val_loss=41.9358 | {'hidden_dim': 512, 'num_layers': 1, 'dropout': 0.3, 'freeze_embeddings': False, 'lr': 0.001, 'weight_decay': 0.0, 'batch_size': 16}
Trial  2 | val_loss=44.8773 | {'hidden_dim': 512, 'num_layers': 1, 'dropout': 0.3, 'freeze_embeddings': False, 'lr': 0.001, 'weight_decay': 0.0, 'batch_size': 16}
Trial  3 | val_loss=54.3110 | {'hidden_dim': 128, 'num_layers': 1, 'dropout': 0.3, 'freeze_embeddings': False, 'lr': 0.0005, 'weight_decay': 0.0005, 'batch_size': 32}
Trial  4 | val_loss=48.6266 | {'hidden_dim': 128, 'num_layers': 2, 'dropout': 0.5, 'freeze_embeddings': False, 'lr': 0.001, 'weight_decay': 0.0005, 'batch_size': 32}
Trial  5 | val_loss=49.8903 | {'hidden_dim': 256, 'num_layers': 2, 'dropout': 0.3, 'freeze_embeddings': False, 'lr': 0.003, 'weight_decay': 0.0, 'batch_size': 16}
Trial  6 | val_loss=43.2976 | {'hidden_dim': 256, 'num_layers': 1, 'dropout': 0.5, 'freeze_embeddings': False, 'lr': 0.0005, 'weight_decay': 0.0001, 'batch_size': 16}
Trial  7 | 

### Retrain best config

In [22]:
best = results_df.iloc[0].to_dict()
print('Best config:')
for k, v in best.items():
    print(f'  {k}: {v}')

CKPT_PATH   = 'model_result/bilstm_crf.pt'
FULL_EPOCHS = 45
PATIENCE    = 5

model = BiLSTM_CRF(
    embedding_matrix  = emb_mat,
    num_tags          = NUM_LABELS,
    pad_token_id      = PAD_ID,
    pad_tag_id        = PAD_LABEL_ID,
    hidden_dim        = int(best['hidden_dim']),
    num_layers        = int(best['num_layers']),
    dropout           = float(best['dropout']),
    freeze_embeddings = bool(best['freeze_embeddings']),
).to(DEVICE)

optimizer   = optim.Adam(model.parameters(), lr=float(best['lr']), weight_decay=float(best['weight_decay']))
t_loader    = DataLoader(train_ds, batch_size=int(best['batch_size']), shuffle=True)
v_loader    = DataLoader(val_ds,   batch_size=int(best['batch_size']), shuffle=False)

best_val, no_imp = float('inf'), 0
train_losses, val_losses = [], []

for ep in range(1, FULL_EPOCHS + 1):
    tr = train_epoch(model, t_loader, optimizer, DEVICE)
    vl = eval_epoch(model, v_loader, DEVICE)
    train_losses.append(tr)
    val_losses.append(vl)
    mark = ''
    if vl < best_val:
        best_val = vl
        torch.save(model.state_dict(), CKPT_PATH)
        no_imp = 0
        mark = ' *'
    else:
        no_imp += 1
    print(f'Epoch {ep:3d}/{FULL_EPOCHS}  train={tr:.4f}  val={vl:.4f}{mark}')
    if no_imp >= PATIENCE:
        print('Early stopping.')
        break

plt.figure(figsize=(8,4))
plt.plot(train_losses, label='train'); plt.plot(val_losses, label='val')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.title('BiLSTM-CRF Loss')
plt.tight_layout(); plt.savefig('model_result/bilstm_crf_loss.png', dpi=100); plt.show()

Best config:
  trial: 1
  val_loss: 41.93583297729492
  hidden_dim: 512
  num_layers: 1
  dropout: 0.3
  freeze_embeddings: False
  lr: 0.001
  weight_decay: 0.0
  batch_size: 16
Epoch   1/45  train=306.6528  val=109.9115 *
Epoch   2/45  train=129.9385  val=91.5150 *
Epoch   3/45  train=102.0946  val=71.2512 *
Epoch   4/45  train=77.5806  val=66.7398 *
Epoch   5/45  train=64.2967  val=50.6813 *
Epoch   6/45  train=47.3328  val=44.8979 *
Epoch   7/45  train=35.7069  val=42.8778 *
Epoch   8/45  train=27.5438  val=42.0511 *
Epoch   9/45  train=22.6386  val=45.9319
Epoch  10/45  train=17.6441  val=46.6465
Epoch  11/45  train=14.6330  val=48.3823
Epoch  12/45  train=12.8925  val=49.1542
Epoch  13/45  train=11.1347  val=50.7223
Early stopping.


### Evaluation

In [23]:
# Load best checkpoint
model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
model.eval()

y_true, y_pred = collect_predictions(model, test_loader, DEVICE, id2label, PAD_LABEL_ID)

# Token accuracy
correct = sum(t == p for t, p in zip(y_true, y_pred))
print(f'Token accuracy : {correct/len(y_true):.4f}  ({correct}/{len(y_true)})')

# Classification report
labels_to_report = [l for l in set(y_true) if l not in ('O', '<PAD_LABEL>')]
print()
print(classification_report(y_true, y_pred, labels=labels_to_report, zero_division=0))

Token accuracy : 0.9066  (6292/6940)

                       precision    recall  f1-score   support

               I-Name       0.80      0.85      0.82        33
B-Companies worked at       0.51      0.50      0.50        52
      B-Email Address       0.73      0.92      0.81        26
       B-College Name       0.83      0.53      0.65        19
             I-Degree       0.89      0.62      0.73        52
               B-Name       0.91      0.94      0.93        34
             B-Degree       0.77      0.48      0.59        21
             I-Skills       0.59      0.32      0.41       345
        B-Designation       0.74      0.47      0.57        66
           I-Location       0.00      0.00      0.00         1
       I-College Name       0.66      0.67      0.67        52
           B-Location       0.00      0.00      0.00         8
             B-Skills       0.83      0.19      0.31        26
B-Years of Experience       0.00      0.00      0.00         1
        I-Design

In [24]:
# Span-level seqeval
try:
    from seqeval.metrics import classification_report as seq_report, f1_score as seq_f1

    def to_seqeval(flat_tags, masks_arr):
        seqs, idx = [], 0
        for m in masks_arr:
            length = int(m.sum())
            seqs.append(flat_tags[idx:idx+length])
            idx += length
        return seqs

    y_true_seq = to_seqeval(y_true, masks_test)
    y_pred_seq = to_seqeval(y_pred, masks_test)

    print('=== Test Set Evaluation (seqeval — entity-level) ===')
    print(seq_report(y_true_seq, y_pred_seq, zero_division=0))

    test_f1 = seq_f1(y_true_seq, y_pred_seq, zero_division=0)
    print(f'Test F1 (seqeval) : {test_f1:.4f}')
except ImportError:
    print('seqeval not installed — pip install seqeval')

=== Test Set Evaluation (seqeval — entity-level) ===
                     precision    recall  f1-score   support

       College Name       0.30      0.32      0.31        19
Companies worked at       0.39      0.38      0.39        52
             Degree       0.56      0.43      0.49        21
        Designation       0.46      0.33      0.39        66
      Email Address       0.70      0.88      0.78        26
    Graduation Year       0.00      0.00      0.00        21
           Location       0.00      0.00      0.00         8
               Name       0.72      0.76      0.74        34
             Skills       0.00      0.00      0.00        27
Years of Experience       0.00      0.00      0.00         1

          micro avg       0.49      0.39      0.43       275
          macro avg       0.31      0.31      0.31       275
       weighted avg       0.40      0.39      0.39       275

Test F1 (seqeval) : 0.4318


### Save

In [25]:
import pickle

results_bilstm = {
    'model_name'   : 'BiLSTM-CRF',
    'best_params'  : best,
    'train_losses' : train_losses,
    'val_losses'   : val_losses,
    'y_true'       : y_true,
    'y_pred'       : y_pred,
}

with open('model_result/results_bilstm_crf.pkl', 'wb') as f:
    pickle.dump(results_bilstm, f)

print('Model saved  -> model_result/bilstm_crf.pt')
print('Results saved -> model_result/results_bilstm_crf.pkl')

Model saved  -> model_result/bilstm_crf.pt
Results saved -> model_result/results_bilstm_crf.pkl
